<a href="https://colab.research.google.com/github/coke-jiyong/Data-Analysis-with-Open-Source/blob/main/%EC%98%A4%ED%94%88%EC%86%8C%EC%8A%A4_%EB%8D%B0%EC%9D%B4%ED%84%B0_%EB%B6%84%EC%84%9D_4%EA%B0%95.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



# 오픈소스 기반 데이터 분석 4강 - 데이터 수집


## 4-1 CSV 파일 읽기

In [5]:
import pandas as pd

## data.csv 파일 읽기
df = pd.read_csv('data.csv', encoding='utf-8', sep=',', header = 0,
                 index_col=None, skiprows=None, nrows=None)
print(df)

           날짜    체중  골격근량  체지방량
0  2025.02.06  64.7  30.0  11.1
1  2025.02.04  64.0  29.3  11.6


## 4-2 JSON 파일 읽기



In [8]:
import json
import pandas as pd

## data.json 파일 출력
with open('data.json', mode='r', encoding='utf-8 ') as f:
    data=json.load(f)
print(data)
## data.json 파일 DataFrame 읽기
df = pd.read_json('data.json', orient='records', encoding='utf-8')
print(df)

{'매출데이터': [{'월': '2025-01', '매출액': 1000000, '비용': 700000, '이익': 300000}, {'월': '2025-02', '매출액': 1200000, '비용': 800000, '이익': 400000}, {'월': '2025-03', '매출액': 1500000, '비용': 900000, '이익': 600000}]}
                                               매출데이터
0  {'월': '2025-01', '매출액': 1000000, '비용': 700000,...
1  {'월': '2025-02', '매출액': 1200000, '비용': 800000,...
2  {'월': '2025-03', '매출액': 1500000, '비용': 900000,...


## 4-3 텍스트 파일 읽기 및 데이터 추출

In [9]:
import re

## 파일(callcenter20250301.log) 오픈 및 읽기
with open('callcenter20250301.log', 'r', encoding='utf-8') as f:
    content = f.read()
## 주민등록번호 패턴 생성
pattern = re.compile(r'(\d{6})-(\d{7})')

## 주민등록번호 마스킹
masked_contents = pattern.sub(r'\1-*******', content)

## 마스킹된 파일(callcenter20250301_masked.log) 오픈 및 쓰기
with open ('callcenter20250301_masked.log', mode='w') as f:
    f.write(masked_contents)

print("주민등록번호 마스킹 완료. 'callcenter20250301_masked.log.txt' 파일로 저장되었습니다.")

주민등록번호 마스킹 완료. 'callcenter20250301_masked.log.txt' 파일로 저장되었습니다.


## 4-4 Open-Meteo의 무료 날씨 API를 통한 특정 지역 온도 조회

In [10]:
import requests
import json

url = "https://api.open-meteo.com/v1/forecast?=&=&current=temperature_2m"
params = {
    "latitude": "37.58638333",
    "longitude": "127.0203333",
    "current": "temperature_2m"
}

try:
    ## URL 및 파라미터 전송
    response = requests.get(url, params=params)
    response.raise_for_status()

    ## JSON 데이터 읽기
    data = response.json()

    print("API 응답:", data)
    print("서울시 종로구의 현재 온도는 : {0}{1} 입니다.".format(data['current']['temperature_2m'], data['current_units']['temperature_2m']))

except requests.exceptions.RequestException as e:
    print(f"API 호출 실패: {e}")
except json.JSONDecodeError as e:
    print(f"JSON 파싱 실패: {e}")

API 응답: {'latitude': 37.6, 'longitude': 127.0, 'generationtime_ms': 0.02944469451904297, 'utc_offset_seconds': 0, 'timezone': 'GMT', 'timezone_abbreviation': 'GMT', 'elevation': 29.0, 'current_units': {'time': 'iso8601', 'interval': 'seconds', 'temperature_2m': '°C'}, 'current': {'time': '2026-10-09T15:45', 'interval': 900, 'temperature_2m': 15.9}}
서울시 종로구의 현재 온도는 : 15.9°C 입니다.


In [11]:
!curl -o google-chrome-stable_current_amd64.deb https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt install ./google-chrome-stable_current_amd64.deb -y
!pip install selenium webdriver_manager

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  136M  100  136M    0     0   275M      0 --:--:-- --:--:-- --:--:--  275M
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Note, selecting 'google-chrome-stable' instead of './google-chrome-stable_current_amd64.deb'
The following additional packages will be installed:
  at-spi2-common at-spi2-core gsettings-desktop-schemas libatk-bridge2.0-0t64
  libatk1.0-0t64 libatspi2.0-0t64 libxcomposite1 libxtst6 session-migration
The following NEW packages will be installed:
  at-spi2-common at-spi2-core google-chrome-stable gsettings-desktop-schemas
  libatk-bridge2.0-0t64 libatk1.0-0t64 libatspi2.0-0t64 libxcomposite1
  libxtst6 session-migration
0 upgraded, 10 newly installed, 0 to remove and 2 not upgraded.
Need to get 331 kB/144 MB of archives.
After this operation, 463 MB of addition

## 4-5 Selenium과 lxml을 이용한 웹 스크래핑

In [12]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service as ChromeService
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from lxml import html
import time

chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument('--headless')               # 브라우저 창 없이 실행
chrome_options.add_argument('--no-sandbox')             # 보안모드 비활성화 (Colab 필수)
chrome_options.add_argument('--disable-dev-shm-usage')  # 메모리 부족 방지 (Colab 필수)
chrome_options.add_argument('--window-size=1920x1080')  # 창 크기 설정(가상)
chrome_options.add_argument('--disable-gpu')            # GPU 가속 비활성화 (일부 환경 안정성)
chrome_options.binary_location = "/usr/bin/google-chrome-stable"  # Colab용 크롬 경로 지정

## 드라이버 실행
driver = webdriver.Chrome(options=chrome_options)

## 사이트 접속
url = 'https://professor.knou.ac.kr/jaehwachung/index.do'
driver.get(url)

## 사이트 접속 대기
time.sleep(2)

## 페이지 제목 출력
page_source = driver.page_source
tree = html.fromstring(page_source)

title_text = tree.xpath('//title/text()')
print(title_text)

## 드라이버 종료
driver.quit()

['컴퓨터과학과 정재화 교수']



# 실습 시나리오

## 공공데이터 포털 가입 및 데이터 신청

- [https://www.data.go.kr](https://www.data.go.kr)
- 한국환경공단 에어코리아 대기오염정보 데이터 신청

In [24]:
import requests
import pandas as pd
import time

# 1. API 기본 URL 및 인증키 설정
url = 'https://apis.data.go.kr/B552845/perDay/price'
api_key = 'a4bbab612a2f75c68cd407f9e823babdaf763ef7e602a2197539fd4d673dc7cd'

# 2. 수집 대상 품종 정보 (코드: 품종명)
varieties = {
    '01': '봄배추',
    '02': '여름배추(고랭지)',
    '03': '가을배추',
    '06': '월동배추'
}

# 수집된 전체 데이터를 담을 리스트
all_price_data = []

print("=== 가락도매시장 배추 가격 데이터 수집 시작 (2015~2024) ===")

# 3. 2015년부터 2024년까지, 4개 품종에 대한 반복 호출
for year in range(2015, 2025):
    for vrty_code, vrty_name in varieties.items():
        # 요청 파라미터 구성
        params = {
            'serviceKey': api_key,
            'pageNo': '1',
            'numOfRows': '200',
            'returnType': 'JSON',
            'cond[exmn_ymd::GTE]': f'{year}0101',
            'cond[exmn_ymd::LTE]': f'{year}1231',
            'cond[se_cd::EQ]': '02',      # 중도매(도매)
            'cond[ctgry_cd::EQ]': '200',  # 채소류
            'cond[item_cd::EQ]': '211',   # 배추
            'cond[grd_cd::EQ]': '04',     # 상품
            'cond[sgg_cd::EQ]': '1101',   # 서울
            'cond[mrkt_cd::EQ]': '0110211', # 가락도매
            'cond[vrty_cd::EQ]': vrty_code  # 품종코드
        }

        try:
            response = requests.get(url, params=params)
            res_data = response.json()

            # 응답 구조 접근
            body = res_data.get('response', {}).get('body', {})
            total_count = body.get('totalCount', 0)
            items = body.get('items', {}).get('item', [])

            # 단일 건만 반환될 경우 리스트 변환 처리
            if isinstance(items, dict):
                items = [items]

            print(f"[{year}년 | {vrty_name}({vrty_code})] totalCount: {total_count}건 수집 완료")

            # 결과 리스트에 품종/연도 정보 추가 후 저장
            for item in items:
                item['target_year'] = year
                item['vrty_name'] = vrty_name
                all_price_data.append(item)

        except Exception as e:
            print(f"[{year}년 | {vrty_code}] 호출 실패: {e}")

        # API 호출 간격 조절 (안정적인 데이터 수집용)
        time.sleep(0.1)

# 4. 수집 완료 데이터 변환 및 확인
df = pd.DataFrame(all_price_data)
print("\n==========================================")
print(f"총 {len(df)}건의 일별 배추 가격 데이터 수집 성공!")
print("==========================================")
print(df.head())

# CSV 파일 저장 (필요 시 주석 해제)
# df.to_csv("seoul_garak_cabbage_2015_2024.csv", index=False, encoding='utf-8-sig')

=== 가락도매시장 배추 가격 데이터 수집 시작 (2015~2024) ===
[2015년 | 봄배추(01)] totalCount: 66건 수집 완료
[2015년 | 여름배추(고랭지)(02)] totalCount: 64건 수집 완료
[2015년 | 가을배추(03)] totalCount: 45건 수집 완료
[2015년 | 월동배추(06)] totalCount: 97건 수집 완료
[2016년 | 봄배추(01)] totalCount: 70건 수집 완료
[2016년 | 여름배추(고랭지)(02)] totalCount: 72건 수집 완료
[2016년 | 가을배추(03)] totalCount: 48건 수집 완료
[2016년 | 월동배추(06)] totalCount: 87건 수집 완료
[2017년 | 봄배추(01)] totalCount: 67건 수집 완료
[2017년 | 여름배추(고랭지)(02)] totalCount: 57건 수집 완료
[2017년 | 가을배추(03)] totalCount: 48건 수집 완료
[2017년 | 월동배추(06)] totalCount: 86건 수집 완료
[2018년 | 봄배추(01)] totalCount: 48건 수집 완료
[2018년 | 여름배추(고랭지)(02)] totalCount: 74건 수집 완료
[2018년 | 가을배추(03)] totalCount: 48건 수집 완료
[2018년 | 월동배추(06)] totalCount: 94건 수집 완료
[2019년 | 봄배추(01)] totalCount: 43건 수집 완료
[2019년 | 여름배추(고랭지)(02)] totalCount: 65건 수집 완료
[2019년 | 가을배추(03)] totalCount: 53건 수집 완료
[2019년 | 월동배추(06)] totalCount: 97건 수집 완료
[2020년 | 봄배추(01)] totalCount: 51건 수집 완료
[2020년 | 여름배추(고랭지)(02)] totalCount: 66건 수집 완료
[2020년 | 가을배추(03)] totalCount: 